# Pair-based statistical arbitrage research

This notebook is the readable research layer. Data loading, eligibility rules and statistical calculations live in reusable Python modules that are also used by the Streamlit frontend.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from pairs_trading.data import load_price_data, load_static_data, price_data_audit
from pairs_trading.universe import eligible_companies, same_industry_pairs
from pairs_trading.statistics import analyze_pair
from pairs_trading.backtest import backtest_pair

## 1. Load a formation period

Change these years and dates for a different historical formation window. No future trading-period data is used in the pair estimation below.

In [ ]:
YEARS = range(2017, 2021)
FORMATION_START = '2017-01-01'
FORMATION_END = '2019-12-31'

prices = load_price_data(PROJECT_ROOT, YEARS)
company_master, static_audit = load_static_data(PROJECT_ROOT)
price_data_audit(prices)

## 2. Construct the eligible universe

Eligibility is based only on observations available inside the formation window.

In [ ]:
eligible = eligible_companies(
    prices, company_master, FORMATION_START, FORMATION_END,
    minimum_coverage=0.95, minimum_observations=500
)
eligible[['company_name', 'sector', 'industry', 'observations', 'coverage']].head(20)

## 3. Generate economically similar candidates

The industry filter narrows the search space. It does not replace cointegration testing.

In [ ]:
eligible.groupby('industry')['capitaline_code'].nunique().sort_values(ascending=False).head(15)

In [ ]:
INDUSTRY = eligible.groupby('industry')['capitaline_code'].nunique().idxmax()
candidates = same_industry_pairs(eligible, INDUSTRY)
candidates.head()

## 4. Analyse one pair

This is an initial formation-period diagnostic. Out-of-sample signal generation, transaction costs, walk-forward backtesting and ML filtering will be added in later stages.

In [ ]:
example = candidates.iloc[0]
metrics, pair_history = analyze_pair(
    prices, int(example.left_code), int(example.right_code),
    FORMATION_START, FORMATION_END
)
metrics

In [ ]:
pair_history.set_index('date')[['z_score']].plot(figsize=(12, 4), title='Formation-period spread z-score', grid=True);

## 5. Out-of-sample baseline backtest

The formation estimates remain frozen. Signals observed at a daily close become positions on the following trading day. The baseline uses entry 2.0, exit 0.5, stop 4.0, a 60-day maximum holding period and 10 basis points of cost per portfolio turnover.

In [ ]:
backtest_metrics, daily_backtest, trade_log = backtest_pair(
    prices, int(example.left_code), int(example.right_code),
    FORMATION_START, FORMATION_END,
    trading_start='2020-01-01', trading_end='2020-06-30',
    entry_z=2.0, exit_z=0.5, stop_z=4.0,
    maximum_holding_days=60, transaction_cost_bps=10.0,
)
{key: backtest_metrics[key] for key in [
    'number_of_trades', 'win_rate', 'net_total_return',
    'annualized_sharpe', 'maximum_drawdown', 'convergence_rate'
]}

In [ ]:
trade_log

In [ ]:
daily_backtest.set_index('date')[['gross_equity', 'net_equity']].plot(
    figsize=(12, 4), title='Out-of-sample portfolio value', grid=True
);